# 04 - Entrenar Modelo 2 (single-label, 5 patogenos) - DOBLE ENTRADA

Transfer learning **EfficientNetB0 (224x224)**, backbone compartido con **dos entradas**: original + hoja aislada. Clasificacion **single-label** (una enfermedad por hoja) con softmax(5).

**Clases**: bacterianas, fungicas, plagas_insectos, roya, virales.
- Loss: CategoricalCrossentropy (label smoothing 0.05). Class weights inverse-frequency. EMA en fine-tune.
- Regularizacion guiada (early stopping/ReduceLR; dropout/aug ajustados por gap train-val).
- Split train/val 80/20 + test independiente (notebook 05). Target: acc/F1 macro > 0.9514/0.9472.

In [ ]:
!pip install -q tensorflow albumentations scikit-learn matplotlib

In [ ]:
import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from pathlib import Path

tf.random.set_seed(42); np.random.seed(42)
print('GPU:', tf.config.list_physical_devices('GPU'))

IMG_SIZE = (224, 224)
BATCH = 16
EPOCHS_P1 = 20
EPOCHS_P2 = 45
LR_P1 = 1e-3
LR_P2 = 1e-5
EMA_DECAY = 0.9999
LABEL_SMOOTH = 0.05
DATA = Path('./splits/clasificacion_patogeno')
OUT = Path('./outputs'); OUT.mkdir(exist_ok=True)

In [ ]:
import time
import numpy as np
import cv2
import albumentations as A
from PIL import Image
from pathlib import Path
from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications.efficientnet import preprocess_input

_HSV_GREEN_LOW  = np.array([20, 25, 30])
_HSV_GREEN_HIGH = np.array([90, 255, 255])
_HSV_LESION_LOW  = np.array([0, 40, 25])
_HSV_LESION_HIGH = np.array([35, 255, 230])
_MORPH = np.ones((7, 7), np.uint8)


def pseudo_leaf_mask(img_rgb):
    bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    leaf = cv2.bitwise_or(cv2.inRange(hsv, _HSV_GREEN_LOW, _HSV_GREEN_HIGH),
                          cv2.inRange(hsv, _HSV_LESION_LOW, _HSV_LESION_HIGH))
    leaf = cv2.morphologyEx(leaf, cv2.MORPH_CLOSE, _MORPH)
    leaf = cv2.morphologyEx(leaf, cv2.MORPH_OPEN, _MORPH)
    return leaf


def leaf_isolate(img_rgb):
    mask = pseudo_leaf_mask(img_rgb)
    out = img_rgb.copy()
    out[mask == 0] = 0
    return out


def safe_read(fp, size):
    for _ in range(3):
        try:
            return np.array(Image.open(fp).convert('RGB').resize(size))
        except (OSError, IOError):
            time.sleep(0.3)
    return None


class DualLeafSequence(Sequence):
    def __init__(self, directory, img_size, batch_size, num_classes, augment, shuffle, aug=None):
        self.img_size = img_size
        self.batch_size = batch_size
        self.num_classes = num_classes
        self.augment = augment
        self.shuffle = shuffle
        self.aug = aug
        self.samples = []
        self.class_indices = {}
        classes = sorted(p.name for p in Path(directory).iterdir() if p.is_dir())
        for i, cls in enumerate(classes):
            self.class_indices[cls] = i
            for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp'):
                for fp in (Path(directory) / cls).glob(ext):
                    self.samples.append((str(fp), i))
        self.n = len(self.samples)
        self.labels = np.array([s[1] for s in self.samples])
        if shuffle:
            self._shuffle()

    def _shuffle(self):
        np.random.shuffle(self.samples)
        self.labels = np.array([s[1] for s in self.samples])

    def __len__(self):
        return max(1, self.n // self.batch_size)

    def __getitem__(self, idx):
        batch = self.samples[idx * self.batch_size:(idx + 1) * self.batch_size]
        orig, leaf, labels = [], [], []
        for fp, cls in batch:
            img = safe_read(fp, self.img_size)
            if img is None:
                continue
            if self.augment and self.aug is not None:
                img = self.aug(image=img)['image']
            iso = leaf_isolate(img)
            orig.append(preprocess_input(img.astype(np.float32)))
            leaf.append(preprocess_input(iso.astype(np.float32)))
            labels.append(cls)
        if not orig:
            z = preprocess_input(np.zeros((1, *self.img_size, 3), np.float32))
            y = np.zeros((1,), np.float32) if self.num_classes == 1 else np.zeros((1, self.num_classes), np.float32)
            return [z, z], y
        X_o = np.stack(orig); X_l = np.stack(leaf)
        if self.num_classes == 1:
            Y = np.array(labels, np.float32)
        else:
            Y = np.eye(self.num_classes, dtype=np.float32)[labels]
        return [X_o, X_l], Y

    def on_epoch_end(self):
        if self.shuffle:
            self._shuffle()


class EMACallback(tf.keras.callbacks.Callback):
    def __init__(self, decay=0.9999):
        super().__init__(); self.decay = decay; self.ema = None
    def on_train_begin(self, logs=None):
        self.ema = [w.numpy().copy() for w in self.model.trainable_weights]
    def on_train_batch_end(self, batch, logs=None):
        for i, w in enumerate(self.model.trainable_weights):
            self.ema[i] = self.decay * self.ema[i] + (1 - self.decay) * w.numpy()
    def on_train_end(self, logs=None):
        for w, e in zip(self.model.trainable_weights, self.ema):
            w.assign(e)


def build_dual(num_classes, backbone, size, l2=5e-5, dropout=(0.3, 0.2)):
    Base = tf.keras.applications.EfficientNetB1 if backbone == 'b1' else tf.keras.applications.EfficientNetB0
    base = Base(weights='imagenet', include_top=False, input_shape=(*size, 3))
    base.trainable = False
    inp_o = tf.keras.layers.Input((*size, 3), name='original')
    inp_l = tf.keras.layers.Input((*size, 3), name='hoja_aislada')
    fo = tf.keras.layers.GlobalAveragePooling2D()(base(inp_o))
    fl = tf.keras.layers.GlobalAveragePooling2D()(base(inp_l))
    x = tf.keras.layers.Concatenate()([fo, fl])
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(dropout[0])(x)
    x = tf.keras.layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(l2))(x)
    x = tf.keras.layers.Dropout(dropout[1])(x)
    act = 'sigmoid' if num_classes == 1 else 'softmax'
    out = tf.keras.layers.Dense(num_classes, activation=act)(x)
    model = tf.keras.Model([inp_o, inp_l], out, name=f'dual_{backbone}')
    model._backbone = base
    return model


def set_finetune(model, last_n):
    base = model._backbone
    base.trainable = True
    for layer in base.layers[:-last_n]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

In [ ]:
TRAIN_AUG = A.Compose([
    A.Rotate(limit=45, border_mode=0, p=0.7),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.6),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.5),
    A.RGBShift(r_shift_limit=15, g_shift_limit=15, b_shift_limit=15, p=0.25),
    A.Affine(translate_percent=0.08, scale=(0.88, 1.15), rotate=0, p=0.4),
])

train_gen = DualLeafSequence(DATA / 'train', IMG_SIZE, BATCH, num_classes=5, augment=True, shuffle=True, aug=TRAIN_AUG)
val_gen = DualLeafSequence(DATA / 'val', IMG_SIZE, BATCH, num_classes=5, augment=False, shuffle=False)
NUM_CLASSES = train_gen.num_classes
CLASS_NAMES = [k for k, _ in sorted(train_gen.class_indices.items(), key=lambda kv: kv[1])]
print(f'Train: {train_gen.n} | Val: {val_gen.n} | Clases: {CLASS_NAMES}')
json.dump(train_gen.class_indices, open(OUT / 'class_indices_model2_pathogen.json', 'w'))

_counts = np.bincount(train_gen.labels, minlength=NUM_CLASSES)
_w = _counts.sum() / (NUM_CLASSES * np.clip(_counts, 1, None))
CLASS_WEIGHT = {i: float(w) for i, w in enumerate(_w)}
print('class_weight:', dict(zip(CLASS_NAMES, [round(w, 3) for w in _w])))

In [ ]:
model = build_dual(num_classes=5, backbone='b0', size=IMG_SIZE)
LOSS = tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH)
METRICS = ['accuracy', tf.keras.metrics.Precision(name='prec'), tf.keras.metrics.Recall(name='rec')]
model.compile(optimizer=tf.keras.optimizers.Adam(LR_P1), loss=LOSS, metrics=METRICS)
model.summary(line_length=110)

In [ ]:
cbs1 = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=6, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(str(OUT / 'model2_pathogen_p1_best.keras'), monitor='val_accuracy', mode='max', save_best_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
]
print('=== FASE 1: cabeza congelada ===')
h1 = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS_P1, callbacks=cbs1, class_weight=CLASS_WEIGHT, verbose=1)

In [ ]:
set_finetune(model, last_n=80)
print('Layers entrenables:', sum(1 for l in model.layers if l.trainable), '+ backbone interno')
steps = max(1, len(train_gen)) * EPOCHS_P2
lr_sched = tf.keras.optimizers.schedules.CosineDecay(LR_P2, steps, alpha=0.01)
model.compile(optimizer=tf.keras.optimizers.Adam(lr_sched), loss=LOSS, metrics=METRICS)
cbs2 = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=12, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(str(OUT / 'model2_pathogen_p2_best.keras'), monitor='val_accuracy', mode='max', save_best_only=True, verbose=1),
    EMACallback(decay=EMA_DECAY),
]
print('=== FASE 2: fine-tune ultimas 80 capas + EMA ===')
h2 = model.fit(train_gen, validation_data=val_gen, epochs=len(h1.history['loss']) + EPOCHS_P2, initial_epoch=len(h1.history['loss']), callbacks=cbs2, class_weight=CLASS_WEIGHT, verbose=1)
model.save(OUT / 'model2_pathogen.keras')
print('guardado', OUT / 'model2_pathogen.keras')

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, classification_report
yt, yp = [], []
for i in range(len(val_gen)):
    x, y = val_gen[i]
    yt.append(np.argmax(y, 1)); yp.append(np.argmax(model.predict(x, verbose=0), 1))
yt = np.concatenate(yt); yp = np.concatenate(yp)
print('val accuracy:', round(accuracy_score(yt, yp), 4), '| F1 macro:', round(f1_score(yt, yp, average='macro', zero_division=0), 4))
print(classification_report(yt, yp, target_names=CLASS_NAMES, zero_division=0))

loss_h = h1.history['loss'] + h2.history['loss']; vloss = h1.history['val_loss'] + h2.history['val_loss']
acc = h1.history['accuracy'] + h2.history['accuracy']; vacc = h1.history['val_accuracy'] + h2.history['val_accuracy']
ep = range(1, len(loss_h) + 1); div = len(h1.history['loss'])
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
a1.plot(ep, loss_h, 'b-', label='train'); a1.plot(ep, vloss, 'r-', label='val'); a1.axvline(div, color='gray', ls='--'); a1.set_title('Loss'); a1.legend(); a1.grid(alpha=0.3)
a2.plot(ep, acc, 'b-', label='train'); a2.plot(ep, vacc, 'r-', label='val'); a2.axvline(div, color='gray', ls='--'); a2.set_title('Accuracy'); a2.legend(); a2.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / 'model2_pathogen_curves.png', dpi=120); plt.show()
gap = vloss[-1] - loss_h[-1]
print('  ALTA VARIANZA: subir dropout/aug.' if gap > 0.15 else ('  ALTO BIAS: mas capacidad/epocas.' if loss_h[-1] > 1.0 else '  Balanceado.'))